# CQCC Precomputation on Kaggle GPU

This notebook extracts Constant-Q Cepstral Coefficients (CQCC) from the ASVspoof2019 LA training set
using `nnAudio` on a Kaggle T4 GPU.

**Time estimate**: ~35 minutes for 25,000 files
**Output**: `cqcc_cache_train.h5` saved to `/kaggle/working/`

After completion, download the HDF5 file and upload to Google Drive:
`Drive/ASVspoof2019_LA/cache/cqcc_cache_train.h5`

Repeat for dev and eval splits using their respective paths.

In [ ]:
# Step 1: Install dependencies
!pip install -q nnAudio librosa h5py tqdm
# Step 2: Import libraries
import torch
import torchaudio
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm
# Step 3: Setup device (GPU on Kaggle)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Step 4: Verify nnAudio CQT layer
from nnAudio.Spectrogram import CQT1992v2
cqt_layer = CQT1992v2(
    sr=16000, hop_length=512, fmin=50, fmax=8000,
    bins_per_octave=36, filter_scale=1.0,
    trainable=False, output_format='Magnitude'
).to(device)
print(f"CQT bins: {cqt_layer.bins}")

In [ ]:
# Step 5: Configure paths
# Upload the ASVspoof2019 LA dataset to Kaggle:
# - Go to kaggle.com/datasets/awsaf49/asvspoof-2019-dataset
# - Click 'Add Dataset' or use the Kaggle API to attach it
# - The folder structure will be: /kaggle/input/asvspoof2019-la/LA_train/flac/

DATA_DIR = Path("/kaggle/input/asvspoof2019-la/LA_train/flac/")
files = sorted(DATA_DIR.glob("*.flac"))
print(f"Found {len(files)} audio files in LA_train")
# Optional: Quick test on 5 files
# test_files = files[:5]

In [ ]:
# Step 6: CQCC Extraction Parameters (defines only, no file handle kept open)
SR = 16000
N_CQCC = 20
HOP_LENGTH = 512
F_MIN = 50
F_MAX = 8000
BINS_PER_OCTAVE = 36
MAX_FRAMES = 300  # Fixed size for HDF5 compatibility

# DCT matrix using librosa (standard cepstral scaling)
import librosa
n_input = cqt_layer.bins
dct_mat = torch.tensor(
    librosa.filters.dct(n_filters=N_CQCC, n_input=n_input),
    dtype=torch.float32, device=device
)
print(f"DCT matrix shape: {dct_mat.shape}")

# HDF5 output path (Kaggle allows writing only to /kaggle/working/)
OUT_PATH = "/kaggle/working/cqcc_cache_train.h5"
print(f"Output will be written to: {OUT_PATH}")


In [ ]:
# Step 7: Main extraction loop with GPU CQT + DCT (FIXED: single open handle)
# FIX: previously dset was created in a closed `with` block, so dset[i] failed.
# Now init + write happen under one open file handle.
with h5py.File(OUT_PATH, 'w') as hf:
    dset = hf.create_dataset(
        "cqcc",
        shape=(len(files), MAX_FRAMES, N_CQCC),
        dtype="float32",
        compression="gzip",
        compression_opts=4,
        chunks=(1, MAX_FRAMES, N_CQCC)
    )
    hf.create_dataset("file_names", data=[f.name for f in files], dtype=h5py.string_dtype())
    hf.attrs["sr"] = SR
    hf.attrs["hop_length"] = HOP_LENGTH
    hf.attrs["n_cqcc"] = N_CQCC
    with torch.no_grad():
        for i, fpath in enumerate(tqdm(files)):
            waveform, sr = torchaudio.load(str(fpath))
            if sr != SR:
                waveform = torchaudio.functional.resample(waveform, sr, SR)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.to(device)
            cqt_mag = cqt_layer(waveform)
            log_cqt = torch.log(cqt_mag.squeeze(0) + 1e-10)
            cqcc = torch.matmul(dct_mat, log_cqt).T
            cqcc_np = cqcc.cpu().numpy()
            pad_width = MAX_FRAMES - cqcc_np.shape[0]
            cqcc_np = np.pad(cqcc_np, ((0, max(pad_width, 0)), (0, 0)), mode='constant')[:MAX_FRAMES, :]
            dset[i] = cqcc_np
            hf.flush()
            if (i + 1) % 500 == 0:
                print(f"Processed {i+1}/{len(files)} files")
print(f"\nExtraction complete! {len(files)} files processed.")


In [ ]:
# Step 8: Verify output
with h5py.File(OUT_PATH, 'r') as hf:
    print(f"CQCC dataset shape: {hf['cqcc'].shape}")
    print(f"Sample CQCC stats: mean={hf['cqcc'][0,:,0].mean():.4f}, std={hf['cqcc'][0,:,0].std():.4f}")
    print(f"File names count: {len(hf['file_names'])}")
    hf.attrs.keys()
# Step 9: Download from Kaggle
# 1. In the output pane on the right, click the 'Download' button next to 'cqcc_cache_train.h5'
# 2. Or use the Kaggle API: kaggle files download -p /kaggle/working/ your-dataset-name
# 3. Upload the .h5 file to Google Drive:
#    Drive/ASVspoof2019_LA/cache/cqcc_cache_train.h5
# Step 10: Repeat for dev and eval splits
# - Dev: DATA_DIR = Path("/kaggle/input/asvspoof2019-la/LA_dev/flac/")
# - Eval: DATA_DIR = Path("/kaggle/input/asvspoof2019-la/LA_eval/flac/")
# - Update OUT_PATH and re-run cells 6-7 with new file lists